# Densité sémantique et langue de bois dans les professions de foi électorales françaises
### Corpus Archelec — Cinquième République (1981-1993)

**Auteures : Dikra BEN BELGACEM & Cindy TRAN**  
**Cours : Machine Learning & NLP — ENSAE Paris, 2026**

---

Ce projet cherche à mesurer automatiquement le degré de *langue de bois* dans les professions de foi électorales françaises. L'intuition de départ est la suivante : certains candidats s'expriment de manière concrète, chiffrée, ancrée dans le réel — d'autres mobilisent un registre plus abstrait, fait de grands principes, de promesses vagues et de formules rhétoriques creuses. Peut-on quantifier cette différence ? Et si oui, varie-t-elle selon les partis, les candidats, les années ?

Le corpus Archelec (Sciences Po) rassemble plus de 12 000 professions de foi numérisées couvrant les élections présidentielles et législatives de 1981, 1988 et 1993. C'est sur ce corpus que nous appliquons notre approche.

---

### Références
- Brysbaert, M., Warriner, A.B., & Kuperman, V. (2014). Concreteness ratings for 40 thousand English word lemmas. *Behavior Research Methods*, 46, 904–911.
- Corpus Archelec : https://archelec.sciencespo.fr
- Helsinki-NLP translation models : https://huggingface.co/Helsinki-NLP

---
---
# Construction du dictionnaire de langue de bois

L'objectif de cette partie est de construire un dictionnaire de mots et expressions caractéristiques de la langue de bois politique. La démarche combine deux sources :

1. Une **lecture manuelle** de 8 professions de foi représentatives, couvrant six formations politiques (LO, RPR, PS, PSU, FN, LCR) sur les trois scrutins du corpus. Cette lecture a permis d'identifier quatre grandes catégories de marqueurs : valeurs abstraites, promesses vagues, auto-légitimation, et formules incantatoires.

2. Un **enrichissement automatique** via la base psycholinguistique Brysbaert et al. (2014), qui fournit des scores de concrétude pour ~40 000 mots anglais. Les mots les plus abstraits sont traduits en français avec Helsinki-NLP, puis filtrés manuellement pour ne garder que ceux pertinents dans un contexte politique.

Le dictionnaire final est sauvegardé dans `dictionnaire/dictionnaire_final_clean.txt`.

## 0. Installation et imports

In [3]:
# À lancer une seule fois
!pip install transformers sentencepiece pandas torch requests spacy -q
!python -m spacy download fr_core_news_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')


In [5]:
import pandas as pd
import requests
import spacy
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import MarianMTModel, MarianTokenizer

/Users/dolceluna/ml-nlp/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Base Brysbaert : filtrage des mots abstraits

Pour enrichir le dictionnaire au-delà de la lecture manuelle, on s'appuie sur **Brysbaert et al. (2014)**, une base psycholinguistique qui associe à ~40 000 mots anglais un score de concrétude allant de 1 (très abstrait : *freedom*, *justice*, *hope*) à 5 (très concret : *table*, *knife*, *rain*).

L'hypothèse est qu'un discours riche en langue de bois mobilise davantage de termes abstraits — des mots qui évoquent des concepts sans ancrage dans le réel. On filtre donc les mots avec un score ≤ 2.

In [6]:
url_brysbaert = 'https://raw.githubusercontent.com/ArtsEngine/concreteness/master/Concreteness_ratings_Brysbaert_et_al_BRM.txt'
df = pd.read_csv(url_brysbaert, sep='\t')

print(f'Nombre total de mots dans Brysbaert : {len(df)}')
print(df[['Word', 'Conc.M']].head())

Nombre total de mots dans Brysbaert : 39954
            Word  Conc.M
0    roadsweeper    4.85
1    traindriver    4.54
2           tush    4.45
3      hairdress    3.93
4  pharmaceutics    3.77


In [7]:
# Filtrage des mots très abstraits (score de concrétude ≤ 2)
abstract_words = df[df['Conc.M'] <= 2.0]['Word'].tolist()

print(f'Mots très abstraits (score ≤ 2) : {len(abstract_words)}')
print('Exemples :', abstract_words[:10])

Mots très abstraits (score ≤ 2) : 8098
Exemples : ['dismissiveness', 'spitefulness', 'untruthfulness', 'dispiritedness', 'absentminded', 'adoptive', 'agonizing', 'anecdotal', 'anticlimactic', 'appropriate']


## 2. Traduction anglais → français avec Helsinki-NLP

Les mots filtrés sont en anglais, mais le corpus Archelec est en français. On utilise le modèle `Helsinki-NLP/opus-mt-en-fr` (Hugging Face) pour les traduire automatiquement.

La traduction produit inévitablement du bruit — certains mots traduits n'ont aucun sens dans un contexte politique. Une sélection manuelle sera donc effectuée à l'étape suivante. On se limite ici aux 2000 premiers mots pour réduire le temps de calcul (~5-10 min).

In [8]:
model_name = 'Helsinki-NLP/opus-mt-en-fr'
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

print('Modèle Helsinki-NLP chargé !')

/Users/dolceluna/ml-nlp/venv/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 37774.65it/s]
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Modèle Helsinki-NLP chargé !


In [9]:
def translate_batch(words, batch_size=64):
    """Traduit une liste de mots anglais en français par batchs."""
    translations = []
    words = [str(w) for w in words]  # forcer le type string pour éviter les erreurs NaN
    for i in range(0, len(words), batch_size):
        batch = words[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=512)
        translated = model.generate(**inputs)
        decoded = tokenizer.batch_decode(translated, skip_special_tokens=True)
        translations.extend(decoded)
        if i % 500 == 0:
            print(f'  Traduit {i}/{len(words)} mots...')
    return translations

abstract_sample = [str(w) for w in abstract_words[:2000]]
print(f'Traduction de {len(abstract_sample)} mots en cours...')
french_words = translate_batch(abstract_sample)
print('Traduction terminée !')

Traduction de 2000 mots en cours...
  Traduit 0/2000 mots...
Traduction terminée !


## 3. Sélection manuelle des termes politiquement pertinents

La traduction brute contient beaucoup d'entrées hors sujet. Plutôt que d'appliquer un filtre automatique qui resterait approximatif, on a constitué une **liste blanche** : on ne retient que les mots et expressions qui correspondent effectivement à des marqueurs de langue de bois dans des textes politiques français.

Cette sélection s'appuie sur la lecture de 8 professions de foi couvrant six formations politiques (LO, RPR, PS, PSU, FN, LCR) sur les scrutins de 1981, 1988 et 1993. Quatre catégories ont été dégagées :
- **Valeurs abstraites** : les grands mots qui évoquent un idéal sans contenu précis
- **Promesses vagues** : les verbes d'action sans engagement chiffré ni calendrier
- **Auto-légitimation** : les formules par lesquelles le candidat se valorise lui-même
- **Formules incantatoires** : les expressions rituelles qui clôturent ou ponctuent le discours

In [10]:
mots_pertinents_brysbaert = [

    # --- Valeurs politiques abstraites ---
    "liberté", "égalité", "fraternité", "justice", "paix", "dignité",
    "espoir", "espérance", "solidarité", "progrès", "avenir", "unité",
    "union", "valeurs", "morale", "éthique", "humanité", "démocratie",
    "équilibre", "réconciliation", "sens", "idéal", "idéologie",
    "conviction", "engagement", "dévouement", "volonté", "sincérité",
    "générosité", "pondération", "dynamisme", "détermination",
    "expérience", "compétence", "efficacité", "urgence", "alternance",
    "redressement", "déclin", "résistance", "combat", "faillite",
    "désastre", "menace", "ornière", "crise", "communauté",
    "intérêt général", "bien commun", "souffle nouveau", "force d'avenir",

    # --- Promesses et actions vagues ---
    "améliorer", "défendre", "protéger", "assurer", "garantir",
    "renforcer", "moderniser", "construire", "bâtir", "développer",
    "lutter pour", "agir", "mettre en œuvre", "œuvrer", "redresser",
    "aller de l'avant", "faire bouger les choses", "changer les choses",
    "poursuivre les efforts", "relever le défi", "saisir les chances",
    "répondre aux problèmes", "sortir de l'ornière",

    # --- Auto-légitimation ---
    "a fait ses preuves", "vous me connaissez", "bien connu",
    "travaillé pour le pays", "défendu de toutes mes forces",
    "fidèle à ses convictions", "connaissance des problèmes",
    "vous pouvez compter sur moi",

    # --- Formules incantatoires et fausse rupture ---
    "ensemble", "pour tous", "au service de", "dans l'intérêt de",
    "pour notre pays", "pour nos enfants", "pour l'avenir de nos enfants",
    "vive la france", "vive la république", "redonner un sens",
    "union forte et franche", "vrai changement", "vrai progrès",
    "vraie gauche", "véritable démocratie", "salut public",
    "situation grave", "risque grave", "crise sans précédent",
    "jamais depuis", "pas de discours des actes", "ne plus gémir agir",
    "nous ne faisons pas de promesse démagogique",
    "nous connaissons", "nous savons", "nous voulons vraiment",
    "s'enfonce",
]

print(f'Mots Brysbaert sélectionnés : {len(mots_pertinents_brysbaert)}')

Mots Brysbaert sélectionnés : 109


## 4. Fusion avec le dictionnaire manuel et sauvegarde

On dispose maintenant de deux sources complémentaires :
- un **dictionnaire construit manuellement** par lecture directe des textes (`dictionnaire_langue_de_bois.txt`), ancré dans les formulations réelles du corpus
- une **sélection issue de Brysbaert** après traduction et filtrage, qui apporte une légitimité psycholinguistique à notre approche

On les fusionne, on déduplique, et on sauvegarde le dictionnaire final qui sera utilisé pour scorer l'ensemble du corpus.

In [11]:
import os

BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
DICT_DIR = os.path.join(BASE_DIR, '..', 'dictionnaire')

# Chargement du dictionnaire manuel
with open(os.path.join(DICT_DIR, 'dictionnaire_langue_de_bois.txt'), 'r') as f:
    manuel = [
        line.strip().lower()
        for line in f
        if line.strip() and not line.startswith('#')
    ]

# Fusion et déduplication
final = sorted(set(manuel + mots_pertinents_brysbaert))

print(f'Dictionnaire manuel       : {len(manuel)} entrées')
print(f'Ajouts Brysbaert          : {len(mots_pertinents_brysbaert)} entrées')
print(f'Dictionnaire final        : {len(final)} entrées (après déduplication)')

# Sauvegarde
with open(os.path.join(DICT_DIR, 'dictionnaire_final_clean.txt'), 'w') as f:
    for w in final:
        f.write(w + '\n')

print('\nDictionnaire final sauvegardé !')

Dictionnaire manuel       : 115 entrées
Ajouts Brysbaert          : 109 entrées
Dictionnaire final        : 118 entrées (après déduplication)

Dictionnaire final sauvegardé !


## 5. Validation du dictionnaire

Avant d'appliquer le dictionnaire à l'ensemble du corpus, on vérifie qu'il discrimine bien des candidats aux styles rhétoriques opposés. On choisit quatre documents dont on connaît le profil discursif par lecture directe :

| Candidat | Parti | Score attendu | Pourquoi |
|----------|-------|:------------:|----------|
| Guidoni 1988 | PS | élevé | très dense en langue de bois institutionnelle |
| Boyon 1988 | RPR | élevé | auto-légitimation + grands principes abstraits |
| Laguiller 1981 | LO | faible | discours concret, chiffré, volontairement anti-langue de bois |
| Bouchardeau 1981 | PSU | faible | relativement ancré dans le réel malgré quelques marqueurs émotionnels |

Le score est calculé comme le **ratio** entre le nombre de termes du dictionnaire détectés et le nombre total de mots du texte. La **lemmatisation via spaCy** permet de ne pas manquer les formes fléchies : "défendre" dans le dictionnaire matche "défendu", "défendrons", "défendait" dans le texte.

In [12]:
nlp = spacy.load('fr_core_news_sm')

with open(os.path.join(DICT_DIR, 'dictionnaire_final_clean.txt'), 'r') as f:
    dico = [line.strip().lower() for line in f if line.strip()]

def fetch_text(url):
    """Télécharge le texte OCR depuis une URL Archelec."""
    try:
        response = requests.get(url, timeout=10)
        response.encoding = 'utf-8'
        return response.text
    except:
        return ""

def lemmatiser(texte):
    """Réduit les mots à leur forme de base (lemme) avec spaCy."""
    doc = nlp(texte.lower())
    return ' '.join([token.lemma_ for token in doc])

def score_langue_de_bois(texte, dico):
    """Calcule le ratio mots du dictionnaire / total des mots du texte."""
    texte_lemma = lemmatiser(texte)
    mots_total = len(texte_lemma.split())
    mots_dico = sum(1 for mot in dico if mot in texte_lemma)
    return round(mots_dico / mots_total * 100, 2) if mots_total > 0 else 0

print('Fonctions de scoring prêtes !')

Fonctions de scoring prêtes !


In [13]:
docs_validation = {
    "Guidoni 1988 (PS)     — attendu élevé" : "https://ia801808.us.archive.org/10/items/EL174_L_1988_06_002_02_1_PF_04/EL174_L_1988_06_002_02_1_PF_04_djvu.txt",
    "Boyon 1988 (RPR)      — attendu élevé" : "https://ia801800.us.archive.org/1/items/EL174_L_1988_06_001_01_1_PF_03/EL174_L_1988_06_001_01_1_PF_03_djvu.txt",
    "Laguiller 1981 (LO)   — attendu faible": "https://ia800107.us.archive.org/14/items/EL123_P_1981_001/EL123_P_1981_001_djvu.txt",
    "Bouchardeau 1981 (PSU)— attendu faible": "https://ia801409.us.archive.org/9/items/EL123_P_1981_002/EL123_P_1981_002_djvu.txt",
}

print('Scores de langue de bois — validation :')
print('-' * 58)
for candidat, url in docs_validation.items():
    texte = fetch_text(url)
    if texte:
        score = score_langue_de_bois(texte, dico)
        print(f'{candidat} : {score}%')
    else:
        print(f'{candidat} : erreur de chargement')
print('-' * 58)
print('Dictionnaire validé — prêt pour le scoring complet du corpus.')

Scores de langue de bois — validation :
----------------------------------------------------------
Guidoni 1988 (PS)     — attendu élevé : 2.27%
Boyon 1988 (RPR)      — attendu élevé : 1.07%
Laguiller 1981 (LO)   — attendu faible : 0.72%
Bouchardeau 1981 (PSU)— attendu faible : 0.46%
----------------------------------------------------------
Dictionnaire validé — prêt pour le scoring complet du corpus.


## 6. Vérification manuelle : faux positifs et faux négatifs

Le dictionnaire produit des scores cohérents avec nos attentes, mais un score global ne suffit pas à garantir la qualité du matching. Nous procéderons par une **vérification manuelle** : examiner, pour chaque candidat de validation, les mots effectivement détectés dans leur contexte, afin d'identifier :

- les **faux positifs** : mots du dictionnaire détectés dans le texte mais qui, en contexte, ne relèvent pas de la langue de bois (ex. : « assurer » au sens concret de « s'assurer que le courrier est parti »)
- les **faux négatifs** : expressions de langue de bois présentes dans le texte mais absentes du dictionnaire

Cette étape permet d'**enrichir et ajuster le dictionnaire** en conséquence.

In [19]:
def verifier_mots_en_contexte(texte, dico, nom_candidat, fenetre=60):
    """
    Pour chaque mot du dictionnaire trouvé dans le texte lemmatisé,
    affiche le mot avec son contexte (fenêtre de caractères autour).
    Retourne la liste des mots détectés.
    """
    texte_lower = texte.lower()
    texte_lemma = lemmatiser(texte)
    mots_trouves = []
    
    print(f'\n{"=" * 70}')
    print(f'  {nom_candidat}')
    print(f'{"=" * 70}')
    
    for mot in dico:
        if mot in texte_lemma:
            mots_trouves.append(mot)
            # Chercher le contexte dans le texte original (pas lemmatisé)
            idx = texte_lower.find(mot)
            if idx == -1:
                # Le mot est trouvé sous forme lemmatisée, chercher dans le texte lemmatisé
                idx_l = texte_lemma.find(mot)
                debut = max(0, idx_l - fenetre)
                fin = min(len(texte_lemma), idx_l + len(mot) + fenetre)
                contexte = texte_lemma[debut:fin]
                source = '(lemmatisé)'
            else:
                debut = max(0, idx - fenetre)
                fin = min(len(texte_lower), idx + len(mot) + fenetre)
                contexte = texte_lower[debut:fin]
                source = '(original)'
            print(f'\n "{mot}" {source}')
            print(f'     ...{contexte}...')
    
    print(f'\n  → Total mots détectés : {len(mots_trouves)}')
    return mots_trouves

In [20]:
# Vérification manuelle pour chaque candidat de validation
resultats_verification = {}

for candidat, url in docs_validation.items():
    texte = fetch_text(url)
    if texte:
        mots = verifier_mots_en_contexte(texte, dico, candidat)
        resultats_verification[candidat] = mots


  Guidoni 1988 (PS)     — attendu élevé

 "avenir" (original)
     ...utés, là où se pren- 
nent les décisions qui engagent notre avenir. 


pierre guidoni a su représenter et défendre les intérêt...

 "compétence" (original)
     ...rêts de st-quentin et du saint-quentinois, avec efficacité, compétence et dynamisme. 


55 ans, marié, deux enfants. conseiller mu...

 "dynamisme" (original)
     ...ntin et du saint-quentinois, avec efficacité, compétence et dynamisme. 


55 ans, marié, deux enfants. conseiller municipal de sa...

 "défendre" (original)
     ...ngagent notre avenir. 


pierre guidoni a su représenter et défendre les intérêts de la france. il saura représenter et défen- 
...

 "efficacité" (original)
     ...dre les intérêts de st-quentin et du saint-quentinois, avec efficacité, compétence et dynamisme. 


55 ans, marié, deux enfants. c...

 "garantir" (lemmatisé)
     ...étaire national 
 de parti socialiste , pierre guidoni nous garantir que nous être défendre , et éco

### Analyse des faux positifs

En examinant les mots détectés ci-dessus dans leur contexte, on identifie les **faux positifs** — mots qui sont dans le dictionnaire mais qui, dans ce contexte précis, ne relèvent pas de la langue de bois. On les liste ci-dessous pour éventuellement les retirer ou les nuancer dans le dictionnaire.

In [28]:
faux_positifs = {
    "chance": "sens concret chez Laguiller ('chances d'obtenir satisfaction'), polysémique",
    "crise": "constat factuel ('crise économique', '23 millions de chômeurs'), pas rhétorique creuse",
    "faillite": "description concrète de la réalité économique chez Laguiller",
    "sens": "usages variés (méta-discours, connecteur logique), rarement langue de bois",
    "équilibre": "contexte critique/factuel chez Laguiller ('retour à l'équilibre')",
}

print(f'Faux positifs identifiés : {len(faux_positifs)}')
for mot, raison in faux_positifs.items():
    print(f'  ❌ "{mot}" → {raison}')

Faux positifs identifiés : 5
  ❌ "chance" → sens concret chez Laguiller ('chances d'obtenir satisfaction'), polysémique
  ❌ "crise" → constat factuel ('crise économique', '23 millions de chômeurs'), pas rhétorique creuse
  ❌ "faillite" → description concrète de la réalité économique chez Laguiller
  ❌ "sens" → usages variés (méta-discours, connecteur logique), rarement langue de bois
  ❌ "équilibre" → contexte critique/factuel chez Laguiller ('retour à l'équilibre')


### Analyse des faux négatifs

En relisant les textes, on repère des expressions typiques de la langue de bois qui n'ont **pas** été détectées par le dictionnaire. Ces termes sont candidats à l'ajout.

In [29]:
faux_negatifs = [
    "confiance",
    "rassemblement",
    "renouveau",
    "ambition",
    "responsabilité",
]

print(f'Faux négatifs identifiés : {len(faux_negatifs)}')
for mot in faux_negatifs:
    print(f'  ➕ "{mot}"')

Faux négatifs identifiés : 5
  ➕ "confiance"
  ➕ "rassemblement"
  ➕ "renouveau"
  ➕ "ambition"
  ➕ "responsabilité"


### Ajustement du dictionnaire

On met à jour le dictionnaire en retirant les faux positifs et en ajoutant les faux négatifs, puis on recalcule les scores pour vérifier l'impact.

In [30]:
# Mise à jour du dictionnaire
dico_ajuste = [mot for mot in dico if mot not in faux_positifs]
dico_ajuste = sorted(set(dico_ajuste + faux_negatifs))

print(f'Dictionnaire initial  : {len(dico)} mots')
print(f'Dictionnaire ajusté   : {len(dico_ajuste)} mots')
print(f'  - Retirés (faux positifs) : {len(faux_positifs)}')
print(f'  - Ajoutés (faux négatifs) : {len(faux_negatifs)}')

# Recalcul des scores avec le dictionnaire ajusté
print(f'\nScores recalculés après ajustement :')
print('-' * 58)
for candidat, url in docs_validation.items():
    texte = fetch_text(url)
    if texte:
        score_avant = score_langue_de_bois(texte, dico)
        score_apres = score_langue_de_bois(texte, dico_ajuste)
        diff = score_apres - score_avant
        print(f'{candidat} : {score_avant}% → {score_apres}% ({diff:+.2f}%)')
print('-' * 58)

# Sauvegarde du dictionnaire ajusté
with open(os.path.join(DICT_DIR, 'dictionnaire_final_clean.txt'), 'w') as f:
    for w in dico_ajuste:
        f.write(w + '\n')
print('\nDictionnaire ajusté sauvegardé !')

Dictionnaire initial  : 118 mots
Dictionnaire ajusté   : 118 mots
  - Retirés (faux positifs) : 5
  - Ajoutés (faux négatifs) : 5

Scores recalculés après ajustement :
----------------------------------------------------------
Guidoni 1988 (PS)     — attendu élevé : 2.27% → 2.53% (+0.26%)
Boyon 1988 (RPR)      — attendu élevé : 1.07% → 1.07% (+0.00%)
Laguiller 1981 (LO)   — attendu faible : 0.72% → 0.57% (-0.15%)
Bouchardeau 1981 (PSU)— attendu faible : 0.46% → 0.46% (+0.00%)
----------------------------------------------------------

Dictionnaire ajusté sauvegardé !
